In [1]:
import polars as pl
import time
start = time.time() 

df = pl.read_csv(
    "../data/raw/complaints.csv",
    columns=[
        'Complaint ID', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
        'Consumer complaint narrative', 'Company', 'State', 'Date received'
    ], 
    ignore_errors=True
)

print(f"Loaded in {time.time()-start:.1f}s — shape: {df.shape}")

Loaded in 882.0s — shape: (9609797, 9)


In [2]:
# Product distribution
print(df['Product'].value_counts())

# Narrative presence
has_narrative = df.filter(pl.col('Consumer complaint narrative').is_not_null()).height
total = df.height
print(f"With narrative: {has_narrative}/{total} ({has_narrative/total*100:.1f}%)")

shape: (21, 2)
┌─────────────────────────────────┬────────┐
│ Product                         ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Bank account or service         ┆ 86205  │
│ Payday loan                     ┆ 5541   │
│ Money transfer, virtual curren… ┆ 145066 │
│ Mortgage                        ┆ 422254 │
│ Credit card or prepaid card     ┆ 206369 │
│ …                               ┆ …      │
│ Credit reporting                ┆ 140429 │
│ Credit card                     ┆ 226686 │
│ Money transfers                 ┆ 5354   │
│ Payday loan, title loan, or pe… ┆ 30641  │
│ Consumer Loan                   ┆ 31574  │
└─────────────────────────────────┴────────┘
With narrative: 2980756/9609797 (31.0%)


In [ ]:
# Product distribution
print(df['Product'].value_counts())

# Narrative presence
has_narrative = df.filter(pl.col('Consumer complaint narrative').is_not_null()).height
total = df.height
print(f"With narrative: {has_narrative}/{total} ({has_narrative/total*100:.1f}%)")

shape: (21, 2)
┌─────────────────────────────────┬────────┐
│ Product                         ┆ count  │
│ ---                             ┆ ---    │
│ str                             ┆ u32    │
╞═════════════════════════════════╪════════╡
│ Virtual currency                ┆ 18     │
│ Credit reporting                ┆ 140429 │
│ Student loan                    ┆ 109717 │
│ Payday loan, title loan, or pe… ┆ 30641  │
│ Other financial service         ┆ 1058   │
│ …                               ┆ …      │
│ Payday loan                     ┆ 5541   │
│ Money transfer, virtual curren… ┆ 145066 │
│ Debt or credit management       ┆ 5047   │
│ Vehicle loan or lease           ┆ 72957  │
│ Prepaid card                    ┆ 15280  │
└─────────────────────────────────┴────────┘
With narrative: 2980756/9609797 (31.0%)


In [6]:
import matplotlib.pyplot as plt

product_counts = df['Product'].value_counts().sort('count', descending=True)
plt.figure(figsize=(12,6))
plt.barh(product_counts['Product'].to_list()[:15], product_counts['count'].to_list()[:15])
plt.title("Complaint Distribution by Product")
plt.tight_layout()
plt.savefig("../notebooks/product_distribution.png", dpi=150)
plt.show()

NameError: name 'df' is not defined

In [5]:
target_products = [
    'Credit card', 'Credit card or prepaid card',
    'Payday loan, title loan, or personal loan', 'Personal loan',
    'Checking or savings account',
    'Money transfer, virtual currency, or service', 'Money transfers'
]

df_filtered = df.filter(
    pl.col('Product').is_in(target_products) &
    pl.col('Consumer complaint narrative').is_not_null()
)

print(f"Filtered shape: {df_filtered.shape}")
print(df_filtered['Product'].value_counts())

NameError: name 'df' is not defined

In [4]:
target_products = [
    'Credit card', 'Credit card or prepaid card',
    'Payday loan, title loan, or personal loan', 'Personal loan',
    'Checking or savings account',
    'Money transfer, virtual currency, or service', 'Money transfers'
]

df_filtered = df.filter(
    pl.col('Product').is_in(target_products) &
    pl.col('Consumer complaint narrative').is_not_null()
)

print(f"Filtered shape: {df_filtered.shape}")
print(df_filtered['Product'].value_counts())

NameError: name 'df' is not defined

In [3]:
product_map = {
    'Credit card': 'Credit Card',
    'Credit card or prepaid card': 'Credit Card',
    'Personal loan': 'Personal Loan',
    'Payday loan, title loan, or personal loan': 'Personal Loan',
    'Checking or savings account': 'Savings Account',
    'Money transfer, virtual currency, or service': 'Money Transfer',
    'Money transfers': 'Money Transfer',
}

df_filtered = df_filtered.with_columns(
    pl.col('Product').replace(product_map).alias('product_category')
)

print(df_filtered['product_category'].value_counts())

NameError: name 'df_filtered' is not defined

In [2]:
import re

def clean_text(text):
    if text is None:
        return ""
    text = text.lower()
    text = re.sub(r'i (am writing|would like) to (file|report|complain|bring).*?\.', '', text)
    text = re.sub(r'\bxx+\b', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

narratives = df_filtered['Consumer complaint narrative'].to_list()
cleaned = [clean_text(t) for t in narratives]

df_filtered = df_filtered.with_columns(pl.Series('clean_narrative', cleaned))

# Drop near-empty after cleaning
df_filtered = df_filtered.filter(pl.col('clean_narrative').str.len_chars() > 50)

print(f"Final shape: {df_filtered.shape}")

NameError: name 'df_filtered' is not defined

In [1]:
import os
os.makedirs("../data/processed", exist_ok=True)

df_filtered.write_csv("../data/processed/filtered_complaints.csv")
print("Saved filtered_complaints.csv ✅")

NameError: name 'df_filtered' is not defined